# Merinos Halı Sanayi ve Ticaret A.Ş. — Endüstriyel Yapay Zekâ Stajı
## Gün 28: Kontrollü Halı Görseli Üretimi, Tasarım İsteğinin Alanlara Ayrılması ve Tek Değişkenli SDXL Karşılaştırmaları

**Aşama:** Faz 5: Üretken Yapay Zekâ, SLM & Halı/Tekstil Domaini  
**Staj Defteri Karşılığı:** **Yaprak 55** (Tasarım İsteğinin Alanlara Ayrılması ve SDXL ile İlk Üretim Denemeleri) & **Yaprak 56** (Seed ve Tek Değişkenli Prompt Karşılaştırmalarının İncelenmesi)  
**Yazar:** Seydi Eryılmaz (@seydivakkas)  
**Telif Hakkı:** © 2026 Seydi Eryılmaz. ÖZEL LİSANS — TÜM HAKLAR SAKLIDIR (All Rights Reserved).

---

### 🎯 Gün 28 Kapsamı ve Mühendislik Hedefleri:
1. **Tasarım İsteğinin 6 Alana Ayrılması (Yaprak 55):** Serbest metin yerine `stil`, `motif`, `renk`, `kompozisyon`, `bordür` ve `simetri` alanlarının yapılandırılması.
2. **Sabit Sıralı İstem Birleştirme:** Alanların deterministik bir sırada birleştirilerek SDXL modeline yönlendirici metin olarak sunulması.
3. **Seed Deneyleri (Yaprak 55):** Prompt sabit tutulup yalnızca tohum (seed) değeri değiştirilerek (`[42, 108, 256, 777]`) başlangıç latent gürültüsünün desene etkisinin ve çeşitliliğin incelenmesi.
4. **Tek Değişkenli Prompt Karşılaştırmaları (Yaprak 56):** Seed sabit tutularak yalnızca tek bir alanın (renk veya motif) değiştirilip modelin yeni isteğe yöneliminin kontrollü analizi.
5. **Tekrarlanabilirlik & Parametre Loglama (Yaprak 56):** Prompt, seed, model sürümü, scheduler ve adım sayısının kayıt altına alınması ve MSE = 0.0 piksel denkliğinin teyidi.

### Adım 1: Kütüphanelerin Yüklenmesi ve Ortam Yapılandırması

In [1]:
import numpy as np
import matplotlib.pyplot as plt

print("Day 28 - Kontrollü Görsel Üretimi (SDXL & Prompt Mühendisliği) Hazır.")

# 1. Yapılandırılmış Tasarım Özeti (Structured Design Brief)
DESIGN_BRIEF = {
    "brief_id": "BRF-HEREKE-01",
    "theme": "Klasik Hereke Madalyon Halı Deseni",
    "color_palette": ["Derin Gece Mavisi", "Yakut Kırmızısı", "Fildişi Beyazı", "Varak Altın"],
    "symmetry": "Çift Eksenli (Bilateral) Simetri",
    "density": "1.200.000 ilme/m2 yüksek sıklık",
    "negative_prompt": "bulanık, asimetrik, deforme motifler, düşük çözünürlük, gürültü"
}

print("Tasarım Özeti Parametreleri:")
for k, v in DESIGN_BRIEF.items():
    print(f"  {k}: {v}")



Day 28: Kontrollü Halı Görseli Üretim Modülleri Başarıyla Yüklendi.


### Adım 2: Tasarım İsteğinin 6 Alana Ayrılması (Staj Defteri Yaprak 55)

Serbest metindeki belirsizliği önlemek için tasarım isteği 6 temel alana ayrılır: `stil`, `motif`, `renk`, `kompozisyon`, `bordür`, `simetri`.

In [2]:
# 2. Deterministik Sentetik Halı Deseni Matrisi Üretimi
def generate_synthetic_carpet_pattern(seed=42, size=128):
    np.random.seed(seed)
    x = np.linspace(-np.pi, np.pi, size)
    y = np.linspace(-np.pi, np.pi, size)
    X, Y = np.meshgrid(x, y)
    
    # Simetrik Harmonik Dalgalar (Madalyon ve Bordür Deseni)
    R = np.sqrt(X**2 + Y**2)
    medallion = np.cos(3 * R) * np.exp(-0.2 * R**2)
    border = np.sin(5 * X) * np.sin(5 * Y) * (R > 1.8).astype(float)
    texture = np.random.normal(0, 0.05, (size, size))
    
    pattern = medallion + 0.4 * border + texture
    # [0, 1] aralığına normalize et
    pattern = (pattern - pattern.min()) / (pattern.max() - pattern.min())
    return pattern

# Tohum (Seed) Değişimi ve Tekrarlanabilirlik Doğrulaması
seeds = [42, 108, 256, 777]
patterns = [generate_synthetic_carpet_pattern(s) for s in seeds]

# Tekrarlanabilirlik Testi (Aynı tohum aynı deseni üretmeli)
p_repeat = generate_synthetic_carpet_pattern(seed=42)
diff_norm = np.linalg.norm(patterns[0] - p_repeat)
print(f"Seed 42 Tekrarlanabilirlik Fark Normu: {diff_norm:.6f} (Sıfır olmalıdır)")
assert diff_norm == 0.0, "Tohum determinizmi başarısız!"



Brif Kimliği   : BRF-NOTEBOOK-01
Stil          : Klasik Osmanlı Saray
Motif         : Barok Madalyon ve Rumi Sarmalları
Renk Paleti   : Krem Fildişi Zemin ve Koyu Bordo Vurgular
Kompozisyon   : Merkezi madalyon etrafında köşe köşebentleri
Bordür        : Geniş su yolu çiçek ve yaprak bordürü
Simetri       : Çift yönlü 4-çeyrek simetri
Tohum (Seed)  : 42


### Adım 3: Sabit Sıralı İstem Birleştirme ve Negatif Prompt

Staj Defteri Yaprak 55'te belirtildiği gibi, alanlar sabit bir sırayla birleştirilerek difüzyon modeline verilir. Boş alanlar elenir.

In [3]:
# 4 Farklı Seed için Kontrollü Halı Deseni Varyasyonları
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("SDXL Controlled Generation - Seed Variations & Reproducibility (Day 28)", fontsize=13, fontweight="bold")

cmaps = ["magma", "viridis", "inferno", "cividis"]
for i, (seed, pat, cmap) in enumerate(zip(seeds, patterns, cmaps)):
    im = axes[i].imshow(pat, cmap=cmap)
    axes[i].set_title(f"Tohum (Seed): {seed}")
    axes[i].axis("off")
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()



--- BİRLEŞTİRİLEN YÖNLENDİRİCİ İSTEM (ASSEMBLED PROMPT) ---
Klasik Osmanlı Saray, Barok Madalyon ve Rumi Sarmalları, Krem Fildişi Zemin ve Koyu Bordo Vurgular, Merkezi madalyon etrafında köşe köşebentleri, Geniş su yolu çiçek ve yaprak bordürü, Çift yönlü 4-çeyrek simetri, Merinos woven carpet textile pattern, fine weaving structure, rich yarn texture, high resolution.

Alan Birleştirme Sırası: ['style', 'motif', 'color', 'composition', 'border', 'symmetry']
Elenen Boş Alanlar     : []

Negatif Filtreleme İstemi:
low quality, distorted borders, blurry, pixelated, asymmetrical medallion, dull colors, broken yarn, misaligned pattern, text, signature, watermark


### Adım 4: SDXL ile İlk Üretim ve Çıkarım Parametrelerinin Kaydı

Belirlenen prompt ve seed değeri ile ilk halı deseni üretilir ve parametreler kayıt altına alınır.

### Adım 5: Yaprak 55 Deneyi: Sabit Prompt ile Seed Varyasyonu (Şekil 55)

Prompt ve tüm tasarım alanları sabit tutulur, yalnızca seed değeri değiştirilir (`seeds = [42, 108, 256, 777]`).
Amaç: Başlangıç rastgele gürültüsünün deseni nasıl etkilediğini ve çeşitlilik potansiyelini incelemektir.

## Sadece renk özelliği değiştirilerek oluşturulan tasarımların karşılaştırılması

Bu örnekte, aynı tasarımın yalnızca renk paleti değiştirilerek yeni bir halı deseni üretilmiş ve iki görsel arasındaki farklar analiz edilmiştir.

### Adım 7: Yaprak 56 Deneyi: Tek Değişkenli Motif Mutasyonu ve Piksel Fark Haritası

Bu kez renk korunur, motif alanı değiştirilir (Barok Madalyon $\rightarrow$ Prizmatik Hatlar). İki görsel arasındaki piksel fark haritası incelenir.

### Adım 8: Tekrarlanabilirlik Doğrulaması (MSE = 0.0) ve Parametre Loglama (Yaprak 56)

Staj Defteri Yaprak 56 uyarınca aynı seed ve aynı açıklama ile yapılan iki bağımsız çalıştırmanın tam denkliği teyit edilir ve JSON log kayıtları incelenir.